## Custom Matrix Mapper

Example of custom mapping.

## Fairy/seed pixel curtain

Similar to these: https://www.youtube.com/watch?v=URwnwcRwbFo


In [1]:
from ledmap.legacy import Base

In [2]:
class Curtain(Base):
    """Fairy/seed pixel curtain."""

    def __init__(
        self,
        width: int = 1,
        height: int = 1,
        delta: int = 1,
    ):
        """Create curtain matrix."""
        assert width > 0
        assert height > 0
        assert delta > 0
        assert height <= delta, "Height must be less than delta"
        self._width = width
        self._height = height
        self._delta = delta

    @property
    def width(self) -> int:
        """Width of the matrix."""
        return self._width

    @property
    def height(self) -> int:
        """Height of the matrix."""
        return self._height

    def mapper(self, x: int, y: int) -> int:
        """Map pixel location to index."""
        return x * self._delta + y

    def repr_args(self) -> tuple[list[str], dict[str, str]]:
        """Get arguments to repr."""
        args, kwargs = super().repr_args()
        kwargs["width"] = repr(self._width)
        kwargs["height"] = repr(self._height)
        kwargs["delta"] = repr(self._delta)
        return args, kwargs

In [3]:
class ExtendedCurtain(Base):
    """Fairy/seed pixel curtain including header.

    Header is where the signal is split for each column. This can be addressed,
    but is oddly-referenced.
    """

    def __init__(
        self,
        width: int = 1,
        height: int = 1,  # natural height rather than total
        delta: int = 1,
    ):
        """Create curtain matrix."""
        assert width > 0
        assert height >= 0
        assert delta > 0
        assert height < delta, "Height must be less than delta"
        self._width = width
        self._height = height
        self._delta = delta

    @property
    def width(self) -> int:
        """Width of the matrix."""
        return self._width

    @property
    def height(self) -> int:
        """Height of the matrix."""
        return self._height + 1

    def mapper(self, x: int, y: int) -> int:
        """Map pixel location to index."""
        return (x + (y <= 0)) * self._delta + y - 1

    def repr_args(self) -> tuple[list[str], dict[str, str]]:
        """Get arguments to repr."""
        args, kwargs = super().repr_args()
        kwargs["width"] = repr(self._width)
        kwargs["height"] = repr(self._height)
        kwargs["delta"] = repr(self._delta)
        return args, kwargs

In [4]:
m = Curtain(5, 5, 10)
m.print()

Curtain(width=5,height=5,delta=10)
   0  10  20  30  40
   1  11  21  31  41
   2  12  22  32  42
   3  13  23  33  43
   4  14  24  34  44


In [5]:
m = ExtendedCurtain(4, 4, 10)
m.print()

ExtendedCurtain(width=4,height=4,delta=10)
   9  19  29  39
   0  10  20  30
   1  11  21  31
   2  12  22  32
   3  13  23  33


## An actual curtain

My hard-coded ESPHome config (including header) contains:

```yaml
  width: 20
  height: 21
  pixel_mapper: |-
    return 799 - 40 * (x + (y >= 0)) + y;
```

In [6]:
from ledmap.legacy import FlipLR, Limit

m = Limit(FlipLR(ExtendedCurtain(20, 20, 40)), last=798)
m.print()

Limit(FlipLR(ExtendedCurtain(width=20,height=20,delta=40)),last=798)
   -1  759  719  679  639  599  559  519  479  439  399  359  319  279  239  199  159  119   79   39
  760  720  680  640  600  560  520  480  440  400  360  320  280  240  200  160  120   80   40    0
  761  721  681  641  601  561  521  481  441  401  361  321  281  241  201  161  121   81   41    1
  762  722  682  642  602  562  522  482  442  402  362  322  282  242  202  162  122   82   42    2
  763  723  683  643  603  563  523  483  443  403  363  323  283  243  203  163  123   83   43    3
  764  724  684  644  604  564  524  484  444  404  364  324  284  244  204  164  124   84   44    4
  765  725  685  645  605  565  525  485  445  405  365  325  285  245  205  165  125   85   45    5
  766  726  686  646  606  566  526  486  446  406  366  326  286  246  206  166  126   86   46    6
  767  727  687  647  607  567  527  487  447  407  367  327  287  247  207  167  127   87   47    7
  768  728  688  648  

In [7]:
from ledmap.util import get_string

print(get_string(m.dump))

{"map":[-1,759,719,679,639,599,559,519,479,439,399,359,319,279,239,199,159,119,79,39,760,720,680,640,600,560,520,480,440,400,360,320,280,240,200,160,120,80,40,0,761,721,681,641,601,561,521,481,441,401,361,321,281,241,201,161,121,81,41,1,762,722,682,642,602,562,522,482,442,402,362,322,282,242,202,162,122,82,42,2,763,723,683,643,603,563,523,483,443,403,363,323,283,243,203,163,123,83,43,3,764,724,684,644,604,564,524,484,444,404,364,324,284,244,204,164,124,84,44,4,765,725,685,645,605,565,525,485,445,405,365,325,285,245,205,165,125,85,45,5,766,726,686,646,606,566,526,486,446,406,366,326,286,246,206,166,126,86,46,6,767,727,687,647,607,567,527,487,447,407,367,327,287,247,207,167,127,87,47,7,768,728,688,648,608,568,528,488,448,408,368,328,288,248,208,168,128,88,48,8,769,729,689,649,609,569,529,489,449,409,369,329,289,249,209,169,129,89,49,9,770,730,690,650,610,570,530,490,450,410,370,330,290,250,210,170,130,90,50,10,771,731,691,651,611,571,531,491,451,411,371,331,291,251,211,171,131,91,51,11,7